# 09 — FlashAttention in Practice

FlashAttention achieves exact attention with O(1) peak memory overhead through IO-aware tiling.

## IO-Aware Algorithm and Practical Usage

The memory bottleneck in standard attention is not the compute — it's the **memory bandwidth**: writing and reading the n×n attention matrix to/from HBM (slow GPU memory).

**FlashAttention** (Dao et al., 2022) tiles the attention computation to fit in SRAM (fast, ~10x bandwidth):
1. Split *Q*, *K*, *V* into blocks that fit in SRAM (~100KB each)
2. Compute attention block by block, maintaining running softmax normalisation via the 'online softmax' algorithm
3. Never materialise the full n×n matrix in HBM

**Result**: exact attention (not approximated) with:
- O(n) HBM reads/writes (vs O(n²) for standard attention)
- 2-4x wall-clock speedup on long sequences
- O(1) peak attention memory (the KV blocks, not the full matrix)

**FlashAttention-2** (Dao, 2023): parallelises across sequence dimension (not just batch/head), reduces non-matmul operations, achieves ~70% A100 utilisation.

**FlashAttention-3**: leverages H100's asynchronous Tensor Core operations.

In practice: use `torch.nn.functional.scaled_dot_product_attention()` which dispatches to FlashAttention when available.

In [ ]:
# FlashAttention benchmarking on long sequences
import torch
import time
import matplotlib.pyplot as plt

# Standard attention
def standard_attention(q, k, v):
    scale = q.shape[-1] ** -0.5
    attn = (q @ k.transpose(-2, -1) * scale).softmax(dim=-1)
    return attn @ v

# PyTorch built-in (uses FlashAttention when available)
def sdpa(q, k, v):
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

# Benchmark
seq_lens = [128, 256, 512, 1024, 2048]
batch, heads, d_head = 2, 8, 64

std_times, flash_times = [], []
std_mem, flash_mem = [], []

for n in seq_lens:
    q = torch.randn(batch, heads, n, d_head)
    k = torch.randn(batch, heads, n, d_head)
    v = torch.randn(batch, heads, n, d_head)

    # Standard attention
    t0 = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            out_std = standard_attention(q, k, v)
    std_times.append((time.perf_counter() - t0) / 10 * 1000)
    std_mem.append(n * n * batch * heads * 2 / 1e6)  # n² attn matrix

    # SDPA (FlashAttention)
    t0 = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            out_flash = sdpa(q, k, v)
    flash_times.append((time.perf_counter() - t0) / 10 * 1000)
    flash_mem.append(n * d_head * batch * heads * 2 / 1e6)  # O(n) memory

    # Verify correctness
    diff = (out_std - out_flash).abs().max().item()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.semilogy(seq_lens, std_times, 'o-', label='Standard', color='tomato')
ax1.semilogy(seq_lens, flash_times, 's-', label='SDPA/Flash', color='steelblue')
ax1.set_xlabel('Sequence length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Attention Speed')
ax1.legend()

ax2.semilogy(seq_lens, std_mem, 'o-', label='Standard O(n²)', color='tomato')
ax2.semilogy(seq_lens, flash_mem, 's-', label='FlashAttn O(n)', color='steelblue')
ax2.set_xlabel('Sequence length')
ax2.set_ylabel('Memory (MB)')
ax2.set_title('Attention Memory')
ax2.legend()

plt.suptitle('Standard vs FlashAttention')
plt.tight_layout()
plt.savefig('/tmp/flash_attention.png', dpi=100, bbox_inches='tight')
plt.show()
print('FlashAttention benchmark saved')